<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

**É necessário normalizar os dados para esse tipo de modelo?**

Não. Random Forest e AdaBoost são modelos baseados em árvores de decisão, que realizam splits binários em limiares de features individuais. Isso os torna **invariantes à escala** dos dados — não importa se os pixels vão de 0 a 255 ou de 0 a 1, as divisões serão as mesmas. Diferente de modelos baseados em distância (como k-NN) ou gradiente (como redes neurais), árvores não se beneficiam de normalização.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

def load_data(seed=42):
    X, y = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto", return_X_y=True)
    y = y.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    return X_train, X_test, y_train, y_test

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(n_estimators=50, random_state=seed, algorithm="SAMME")
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc

A função `evaluate` recebe um modelo já treinado e os dados de teste, realiza as predições com `model.predict()` e retorna a acurácia calculada com `accuracy_score`. A acurácia mede a proporção de predições corretas sobre o total de amostras.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError(f"model_type deve ser 'rf' ou 'ab', recebido: {model_type}")
    acc = evaluate(model, X_test, y_test)
    return acc

**Em qual profundidade começa o overfitting?**

Em datasets como o MNIST, o overfitting em árvores de decisão tipicamente começa a se manifestar a partir de profundidades em torno de 15-20, quando a árvore começa a memorizar padrões específicos do treino em vez de aprender generalizações.

**Por que a árvore consegue 100% no treino quando max_depth=None?**

Quando `max_depth=None`, a árvore cresce até que cada folha contenha apenas amostras de uma única classe (ou até não haver mais features para dividir). Isso significa que a árvore **memoriza completamente** os dados de treino, criando regras específicas para cada amostra — resultando em 100% de acurácia no treino, mas potencialmente com baixa generalização no teste.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

rf_acc = evaluate(rf_model, X_test, y_test)
ab_acc = evaluate(ab_model, X_test, y_test)

print("=== Random Forest ===")
print(f"Acurácia: {rf_acc:.4f}")
print(classification_report(y_test, rf_model.predict(X_test)))

print("\n=== AdaBoost ===")
print(f"Acurácia: {ab_acc:.4f}")
print(classification_report(y_test, ab_model.predict(X_test)))

if rf_acc > ab_acc:
    print(f"\nRandom Forest teve melhor desempenho ({rf_acc:.4f} vs {ab_acc:.4f})")
else:
    print(f"\nAdaBoost teve melhor desempenho ({ab_acc:.4f} vs {rf_acc:.4f})")

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
seeds = [42, 7]
for seed in seeds:
    acc_rf = run_pipeline("rf", seed=seed)
    acc_ab = run_pipeline("ab", seed=seed)
    print(f"Seed {seed}: RF={acc_rf:.4f}, AdaBoost={acc_ab:.4f}")


acc1 = run_pipeline("rf", seed=42)
acc2 = run_pipeline("rf", seed=42)
print(f"\nReprodutibilidade (RF, seed=42): run1={acc1:.4f}, run2={acc2:.4f}")
print(f"Diferença: {abs(acc1 - acc2):.10f}")
print(f"Reprodutível: {abs(acc1 - acc2) < 1e-6}")

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train))
rf_test_acc = evaluate(rf_model, X_test, y_test)

ab_train_acc = accuracy_score(y_train, ab_model.predict(X_train))
ab_test_acc = evaluate(ab_model, X_test, y_test)

print("=== Análise de Overfitting ===")
print(f"Random Forest - Treino: {rf_train_acc:.4f}, Teste: {rf_test_acc:.4f}, Gap: {rf_train_acc - rf_test_acc:.4f}")
print(f"AdaBoost      - Treino: {ab_train_acc:.4f}, Teste: {ab_test_acc:.4f}, Gap: {ab_train_acc - ab_test_acc:.4f}")

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

print("=== Random Forest - Variando n_estimators ===")
for n in [10, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test))
    print(f"  n_estimators={n:>3d}: Acurácia={acc:.4f}")

print("\n=== AdaBoost - Variando n_estimators ===")
for n in [10, 25, 50, 100]:
    ab = AdaBoostClassifier(n_estimators=n, random_state=42, algorithm="SAMME")
    ab.fit(X_train, y_train)
    acc = accuracy_score(y_test, ab.predict(X_test))
    print(f"  n_estimators={n:>3d}: Acurácia={acc:.4f}")

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [ ]:
print("""
1. A acurácia é suficiente para avaliar os modelos?
Não. A acurácia sozinha não captura desbalanceamento entre classes. Métricas como precisão,
recall e F1-Score são essenciais para entender o desempenho por classe. No MNIST as classes
são relativamente balanceadas, mas em cenários reais isso raramente acontece.

2. Como você garante que o resultado não ocorreu por acaso?
Usando random_state fixo em todas as etapas (split dos dados e treinamento do modelo), o
experimento é determinístico e reprodutível. Além disso, testar com diferentes seeds e
verificar que os resultados são consistentes (variação pequena) demonstra robustez.
Cross-validation seria ainda mais robusto para estimar o desempenho real.

3. Cite dois possíveis problemas metodológicos neste experimento.
(a) Usamos apenas um split treino/teste fixo — cross-validation daria estimativas mais
confiáveis do desempenho real do modelo.
(b) Não houve busca sistemática de hiperparâmetros (ex: GridSearchCV), então os modelos
podem não estar otimizados de forma justa para comparação.

4. O pipeline implementado é confiável? Justifique.
O pipeline é razoavelmente confiável para um experimento inicial: é reprodutível (via
random_state), modular (funções separadas para cada etapa) e gera métricas múltiplas.
Porém, para uso em produção, seria necessário adicionar validação cruzada, busca de
hiperparâmetros e análise de variância entre múltiplas execuções para garantir robustez.
""")